In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv('data_for_ml.csv')

# View dataset
df.head()

,Hours_Studied,Sleep_Hours,Internet_Usage,Gender,Study_Method,Attendance,Final_Score
0,2.0,6.0,3.0,Male,Self Study,75.0,55.0
1,3.0,7.0,2.0,Female,Coaching,80.0,60.0
2,4.0,6.0,4.0,male,Self study,78.0,62.0
3,5.0,8.0,3.0,Femle,Group Study,85.0,70.0
4,6.0,7.0,5.0,Male,Self Study,88.0,72.0


In [3]:
print("Shape:", df.shape)
print("\nInfo:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

Shape: (57, 7)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Hours_Studied   56 non-null     float64
 1   Sleep_Hours     56 non-null     float64
 2   Internet_Usage  56 non-null     float64
 3   Gender          56 non-null     object 
 4   Study_Method    56 non-null     object 
 5   Attendance      56 non-null     float64
 6   Final_Score     56 non-null     float64
dtypes: float64(5), object(2)
memory usage: 3.2+ KB

Missing Values:
Hours_Studied     1
Sleep_Hours       1
Internet_Usage    1
Gender            1
Study_Method      1
Attendance        1
Final_Score       1
dtype: int64


In [4]:
# Fill numeric columns with mean
df = df.fillna(df.mean(numeric_only=True))

# Fill categorical columns with mode
df = df.fillna(df.mode().iloc[0])

print("Missing values after treatment:")
print(df.isnull().sum())

Missing values after treatment:
Hours_Studied     0
Sleep_Hours       0
Internet_Usage    0
Gender            0
Study_Method      0
Attendance        0
Final_Score       0
dtype: int64


In [5]:
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (32, 7)


In [6]:
label_encoders = {}

for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

df.head()

,Hours_Studied,Sleep_Hours,Internet_Usage,Gender,Study_Method,Attendance,Final_Score
0,2.0,6.0,3.0,2,3,75.0,55.0
1,3.0,7.0,2.0,0,0,80.0,60.0
2,4.0,6.0,4.0,3,4,78.0,62.0
3,5.0,8.0,3.0,1,1,85.0,70.0
4,6.0,7.0,5.0,2,3,88.0,72.0


In [7]:
# Assuming last column is target
target_column = df.columns[-1]

features = df.columns[:-1]

print("Target:", target_column)
print("Features:", list(features))

Target: Final_Score
Features: ['Hours_Studied', 'Sleep_Hours', 'Internet_Usage', 'Gender', 'Study_Method', 'Attendance']


In [8]:
scaler = StandardScaler()

In [11]:
results = []

for feature in features[:5]:  # First 5 features
    
    X = df[[feature]]
    y = df[target_column]
    
    # Standardization
    X_scaled = scaler.fit_transform(X)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )
    
    # Model training
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Prediction
    y_pred = model.predict(X_test)
    
    # Evaluation
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Feature": feature,
        "MSE": mse,
        "R2 Score": r2
    })

In [12]:
results_df = pd.DataFrame(results)

# Sort by best R²
results_df = results_df.sort_values(by="R2 Score", ascending=False)

results_df

,Feature,MSE,R2 Score
0,Hours_Studied,15.170253,0.935362
1,Sleep_Hours,42.664800,0.818211
2,Internet_Usage,71.814680,0.694007
4,Study_Method,275.328817,-0.173140
3,Gender,298.617698,-0.272371


In [13]:
best_feature = results_df.iloc[0]

print("Best Feature Based on R² Score:")
print(best_feature)

Best Feature Based on R² Score:
Feature     Hours_Studied
MSE             15.170253
R2 Score         0.935362
Name: 0, dtype: object
